# Debugging Weak Per-Category AP: `thick_wire` / `thin_wire` / `gate_pad`

**Purpose of this notebook:** a chronological, evidence-backed log of the
2026-07-19 investigation into why the 3 newly-added bondwire categories
(`thick_wire`, `thin_wire`, `gate_pad`) scored much worse (AP 0.19-0.37) than the
5 original categories (AP 0.91-0.97) after the first full `8cat_bondwire`
production training run (job `237965304`) completed successfully.

**TL;DR of the finding:** the poor AP was **not** primarily an object-geometry
problem (thin/elongated bond wires being hard to segment) -- it was a genuine,
100%-scale **data-pipeline bug**. The merged dataset's 662 new-category source
photos were each split into 3 separate COCO image entries (one per category),
each carrying ONLY its own category's annotations -- silently asserting the
other 2 categories are absent from a photo where they are actually visibly
present. This created systematically contradictory negative supervision across
*all* of the new-category training data. Fixed by merging the duplicate entries
back into one multi-category image record per photo, with zero annotation loss.

**Where the evidence lives (persists on HPC, doesn't need re-running):**
- Per-category eval summary: `experiments/8cat_bondwire/per_category_eval_summary.json`
- Mask overlay spot-checks: `experiments/8cat_bondwire/mask_spot_check/*.png`
- Source dataset: `data/bent-wire/Thin_multicat/` (+ `metadata.json`)
- Merged dataset: `sam3_data/5_plus_bondwire_dataset/{train,test}/annotation_merged.json`
- All code changes are committed on branch `fix/loss-none-pair-crash`

**Current status:** fix implemented and verified (zero annotation loss). A fresh
training run on the corrected annotations is the next step to confirm the fix
actually improves the 3 categories' AP -- see Open Items at the end.

## Timeline at a glance

| Step | What happened | Tool / finding |
|---|---|---|
| 1 | Production job `237965304` completed all 10 epochs cleanly (no NaN/collapse) | Confirms the 2026-07-18 GradScaler fix holds for a full run |
| 2 | Trainer's built-in `val_pin/detection` meter only reports one AGGREGATE AP (0.657) across all 8 categories | Built `.github/scripts/per_category_coco_eval.py` to get a real per-category breakdown |
| 3 | Per-category breakdown: 5 original categories AP 0.91-0.97; `thick_wire`/`thin_wire`/`gate_pad` AP 0.19-0.37; `war_crack` has 0 test instances | Ruled out "just needs more data" (thick_wire has MORE train instances than some strong categories) |
| 4 | Geometry check: `thick_wire`/`thin_wire` are extremely elongated (aspect ratio 10-19x); `gate_pad` is NOT elongated but still scores badly | Elongation doesn't explain `gate_pad`'s failure -- needed a different theory |
| 5 | Confidence check: max prediction score for all 3 new categories caps at ~0.30-0.35 vs `et_pin`'s 0.96 | Points at a training/calibration issue, not just IoU sensitivity |
| 6 | Visual spot-check (`spot_check_category_masks.py`): GT mask overlays show OTHER bondwire features clearly visible but completely unlabeled in sample images | First direct evidence of a real data problem, not just "hard geometry" |
| 7 | Traced to source: `Thin_multicat/metadata.json` confirms composite masks genuinely contain all 3 category values (`Thick=1, Thin=2, Gatepads=3`) per photo | The source data has the info; something downstream is dropping it |
| 8 | Confirmed `split_multiclass_masks.py` is correct (extracts all 3 categories per photo, working as designed) | Ruled out the Phase-1 splitter as the culprit |
| 9 | Found the real bug: each photo ends up as 3 SEPARATE COCO image entries (`g_`/`h_`/`i_` prefixes), each with only its own category's annotations -- verified at scale: ALL 662 photos, 0 exceptions | **Root cause confirmed** |
| 10 | Fix implemented: `merge_duplicate_photo_entries.py` merges each group into one multi-category image entry | Verified zero annotation loss; config updated to use the fixed files |

## Step 1 -- Production training completed cleanly

Job `237965304` (batch_size=4, no debug instrumentation, the GradScaler/bf16 fix
from 2026-07-18 active) ran all 10 epochs to completion:

```
Successfully completed.
Run time: 29633 sec. (~8h14m)
```

Zero `WEIGHT-NaN` hits and zero zero-loss collapses across the entire run --
confirming the GradScaler fix holds for a full production run, not just the
debug-settings validation window from the previous investigation.

Final aggregate validation (epoch 9, the last epoch):

```
coco_eval_segm_AP:     0.6567
coco_eval_segm_AP_50:  0.7101
coco_eval_segm_AP_75:  0.6866
coco_eval_segm_AR_maxDets@100: 0.8890
```

This looked like a healthy, successful training run at a glance -- but this
number is an AGGREGATE across all 8 categories, which hides a very uneven
per-category picture (see Step 2).

## Step 2 -- Built a per-category AP breakdown tool

The trainer's built-in `val_pin/detection` meter (`sam3.eval.coco_eval_offline.CocoEvaluatorOfflineWithPredFileEvaluators`)
only reports ONE aggregate AP across all categories combined -- there was no
existing way to see per-category numbers without re-running inference.

Added `.github/scripts/per_category_coco_eval.py`: reuses the predictions
already dumped by the last validation epoch
(`dumps/pins/coco_predictions_segm.json`, written by
`sam3.eval.coco_writer.PredictionDumper` with `merge_predictions: True`) plus
the ground-truth test annotation file, and runs `pycocotools.COCOeval` once per
category (by restricting `params.catIds`) to get a real per-category AP/AR
breakdown. **No new inference run needed** -- pure re-analysis of predictions
already computed and saved during training.

```bash
python .github/scripts/per_category_coco_eval.py \
  --gt sam3_data/5_plus_bondwire_dataset/test/annotation.json \
  --pred experiments/8cat_bondwire/dumps/pins/coco_predictions_segm.json \
  --out experiments/8cat_bondwire/per_category_eval_summary.json
```

## Step 3 -- Per-category results: 5 strong, 3 weak, 1 unscoreable

| Category | GT instances (test) | AP | AP_50 |
|---|---|---|---|
| et_pin | 1475 | 0.913 | 0.992 |
| eberhard_pin | 1000 | 0.969 | 1.000 |
| wuxi_pin | 1538 | 0.955 | 1.000 |
| matrix_chip | 2853 | 0.961 | 0.990 |
| war_crack | **0** | -- | -- |
| **thick_wire** | 636 | **0.369** | 0.384 |
| **thin_wire** | 277 | **0.242** | 0.290 |
| **gate_pad** | 226 | **0.188** | 0.316 |

Two distinct issues immediately visible:

1. **`war_crack` has a train/test split gap**: 1000 training instances but ZERO
   test instances -- every annotation ended up in the train split. Separate
   issue, tracked independently (not the focus of this notebook).
2. **The 3 new bondwire categories are all underperforming**, and NOT simply
   due to less training data: `thick_wire` (4810 train instances) and
   `gate_pad` (1843) actually have MORE training data than `eberhard_pin`/
   `war_crack` (1000 each), which score near-perfectly. Something else is going
   on.

## Step 4 -- Geometry check: elongation explains 2 of 3, not all 3

Computed average bbox aspect ratio (`max(w,h)/min(w,h)`) per category from the
training annotations:

| Category | avg area (px) | avg aspect ratio | AP |
|---|---|---|---|
| et_pin (control) | 549 | 1.07 | 0.913 |
| matrix_chip | 12296 | 1.68 | 0.961 |
| war_crack | 2805 | 4.85 | (unscoreable) |
| **thick_wire** | 25827 | **19.37** (\~700x37px) | 0.369 |
| **thin_wire** | 6537 | **10.25** (\~234x26px) | 0.242 |
| **gate_pad** | 943 | 1.78 | 0.188 |

`thick_wire`/`thin_wire` are extremely thin, elongated masks -- a well-known
hard case for IoU-based mask losses (a few pixels of boundary error wipes out a
much larger fraction of a thin object's area than the same error on a compact
blob). `AP_small=0.0` for both is consistent with this.

But `gate_pad`'s aspect ratio (1.78) is essentially the SAME as `matrix_chip`'s
(1.68, AP=0.961) -- not elongated at all, yet it still scores badly (0.188).
**Geometry alone cannot explain `gate_pad`'s failure.** Needed a different
theory.

## Step 5 -- Confidence check: all 3 categories are uniformly under-confident

Checked the prediction confidence score distribution per category from the
dumped predictions:

| Category | n_preds | mean score | max score |
|---|---|---|---|
| et_pin (control) | 124600 | 0.013 | **0.957** |
| thick_wire | 4803 | 0.137 | **0.354** |
| thin_wire | 24351 | 0.026 | **0.328** |
| gate_pad | 4740 | 0.046 | **0.303** |

`et_pin`'s BEST prediction reaches 0.96 confidence; all three new categories
cap out around 0.30-0.35 even on their single best prediction. This is a
calibration/under-training signal on the classification head specific to these
categories -- not just a mask-boundary/IoU problem. Combined with `gate_pad`'s
geometry not explaining its failure, this pointed toward something structural
in how these 3 categories are represented in the training data.

## Step 6 -- Visual spot-check: the smoking gun

Added `.github/scripts/spot_check_category_masks.py`: for a given category,
picks sample images containing it and renders the GT polygon mask outline +
bbox for EVERY annotation present in that image's COCO entry, overlaid on the
source photo.

```bash
python .github/scripts/spot_check_category_masks.py \
  --ann sam3_data/5_plus_bondwire_dataset/train/annotation.json \
  --img-dir sam3_data/5_plus_bondwire_dataset/train/images \
  --categories thick_wire thin_wire gate_pad \
  --num-samples 3 \
  --out-dir experiments/8cat_bondwire/mask_spot_check
```

**Result: every single sample showed clearly-visible OTHER bondwire features
with zero annotation.** For example:
- A `thick_wire`-labeled image showed 8 correctly-outlined zigzag wires, PLUS 2
  additional curved/looped wires with no outline at all.
- A `thin_wire`-labeled image showed 2 correctly-outlined thin wires, PLUS an
  entire row of zigzag `thick_wire`-pattern wires with zero annotation.
- A `gate_pad`-labeled image showed 4 correctly-boxed gate pads, in a photo
  PACKED with unlabeled zigzag wires and curved thin wires.

This ruled out "hard geometry" as the primary explanation -- these are missing
labels, not boundary-precision failures. The question became: is this a
genuine source-data gap, or a bug in our own conversion pipeline?

## Step 7 -- Traced to source: the composite masks DO have all 3 categories

Checked the raw source dataset (`data/bent-wire/Thin_multicat/`) directly.
Each photo has a companion `_mask.png` composite label mask. Checked the pixel
value histogram of one source mask:

```
shape: (600, 700), unique values: 0 (bg), 1, 2, 3
```

And `metadata.json` in the same directory:

```json
"categories": [{"id": 1, "name": "Thick"}, {"id": 2, "name": "Thin"}, {"id": 3, "name": "Gatepads"}],
"overlap_rule": "priority"
```

**Confirmed: the source composite mask genuinely contains all 3 category
values in the same photo.** The information is NOT missing from the source
data -- it's being lost or mishandled somewhere in our own conversion
pipeline.

## Step 8 -- Ruled out the Phase-1 mask-splitting script

Read `sam3_fine_tune/preprocessing/split_multiclass_masks.py` in full. Its core
loop:

```python
for label_value, category_name in label_map.items():
    binary = np.where(mask == label_value, 255, 0).astype(np.uint8)
    if not binary.any():
        continue  # this image doesn't contain this category
    # ... copies the image + this category's binary mask into
    # <output_root>/<category_slug>/{images,masks}/
```

This is **correct**: for a photo whose composite mask has values `{1, 2, 3}`,
this script copies the image into ALL THREE of `thick wire/`, `thin wire/`,
and `gate pad/` staging folders, each with its own correctly-isolated binary
mask. The splitter is working exactly as designed -- **not a bug in this
script**. The problem had to be further downstream, in COCO conversion or the
merge-append step.

## Step 9 -- ROOT CAUSE CONFIRMED: split-photo contradictory negatives

Checked whether the SAME underlying photo appears under more than one COCO
image entry in the final merged `train/annotation.json`. Searching by base
filename (stripping the `g_`/`h_`/`i_` prefix) for one sample photo:

```
g_00583722_..._original_multicat.png | categories: {6} (thick_wire) | 8 anns
h_00583722_..._original_multicat.png | categories: {7} (thin_wire)  | 4 anns
i_00583722_..._original_multicat.png | categories: {8} (gate_pad)   | 4 anns
```

**The exact same photo appears 3 times**, as 3 separate COCO `image` entries
(differing only by the `g_`/`h_`/`i_` filename prefix added during COCO
conversion to disambiguate each category's copy), each carrying ONLY its own
category's annotations.

**Verified this holds at scale, not just for one sample:**

```
total unique base photos (g/h/i prefixed): 662
appear in all 3 categories:   662   <-- every single one
appear in exactly 2 categories: 0
appear in exactly 1 category:   0
```

**Why this matters:** the trainer's data loader uses
`include_negatives=True` (`COCO_FROM_JSON`), meaning any category NOT
annotated on a given image entry is treated as a confirmed-absent negative
example for that image. So when the `g_...` (thick_wire) entry is loaded
during training, the model is taught "there is no thin_wire and no gate_pad in
this image" -- even though the exact same pixels, one entry over, are
correctly labeled WITH those objects. Every one of the 662 photos generates 3
mutually-contradictory training samples for these 3 categories. This
explains **all** of the earlier evidence:
- Why confidence caps low (Step 5): the model is being actively taught to
  suppress these categories about 2/3 of the time it sees them.
- Why `gate_pad` fails despite compact geometry (Step 4): geometry was never
  the real driver for it.
- Why every spot-checked sample showed unlabeled sibling features (Step 6):
  they WERE labeled -- just in a sibling image entry sharing the same photo.

## Step 10 -- The fix: merge duplicate-photo entries

Added `.github/scripts/merge_duplicate_photo_entries.py`: groups all COCO
image entries by base filename (stripping the `g_`/`h_`/`i_` prefix), and for
each group with more than one entry, keeps a single canonical entry and
remaps every annotation's `image_id` to point at it -- so each of the 662
photos ends up as ONE image entry carrying the union of all 3 categories'
annotations, instead of 3 contradictory single-category copies.

```bash
python .github/scripts/merge_duplicate_photo_entries.py \
  --ann sam3_data/5_plus_bondwire_dataset/train/annotation.json \
  --out sam3_data/5_plus_bondwire_dataset/train/annotation_merged.json
```

**Verified zero annotation loss** (only image-entry bookkeeping changed, no
annotation content lost):

```
                 train              test
images before:   7234               3670
images after:    5910 (-1324)       3444 (-226)
annotations:     24666 (unchanged)  8005 (unchanged)
merged groups:   662                113
```

Spot-checked the same sample photo post-fix -- now correctly ONE entry with
ALL 3 categories combined:

```
g_00583722_..._original_multicat.png | categories: {6, 7, 8} | 16 anns
```

`full_train_config.yaml`'s train/val/eval `ann_file`/`gt_path` fields now
point at the new `annotation_merged.json` files. The original `annotation.json`
files are left untouched -- this is a **non-destructive** fix; the merged
files are new, separate outputs.

## Reusable check: verify a photo's merged categories

Quick reusable snippet to spot-check any base photo's categories before/after
the merge, without needing a new training run.

In [ ]:
import json
from pathlib import Path

DATA_ROOT = Path("/home/hpc_storage/data/projects/userB/sam3_data/5_plus_bondwire_dataset")


def show_photo_categories(ann_path: Path, base_photo_suffix: str):
    """Print which category ids/annotation counts a given base photo has in
    the given annotation file (works for both the original, duplicated
    annotation.json and the fixed annotation_merged.json).
    """
    data = json.load(open(ann_path))
    cat_names = {c["id"]: c["name"] for c in data["categories"]}
    matches = [img for img in data["images"] if img["file_name"].endswith(base_photo_suffix)]
    print(f"{ann_path.name}: {len(matches)} image entrie(s) for this photo")
    for img in matches:
        anns = [a for a in data["annotations"] if a["image_id"] == img["id"]]
        cats = sorted({cat_names[a["category_id"]] for a in anns})
        print(f"  {img['file_name']} -> categories={cats}, num_anns={len(anns)}")


sample_base = (
    "00583722_Image_81901039_2482402244951622_Welder_1.1_20241108_065712_"
    "FS01_FS01ENG_0_0_Block1UpperLeft_original_multicat.png"
)

print("=== BEFORE (original, duplicated) ===")
show_photo_categories(DATA_ROOT / "train" / "annotation.json", sample_base)
print()
print("=== AFTER (fixed, merged) ===")
show_photo_categories(DATA_ROOT / "train" / "annotation_merged.json", sample_base)

## How to re-check per-category AP after the fix-validation training run

```bash
ssh hpc_muc '
REPO=/home/hpc_storage/data/projects/userB/test_sam3/sam3_fine_tune
DATA=/home/hpc_storage/data/projects/userB/sam3_data/5_plus_bondwire_dataset
cd "$REPO" && source .venv/bin/activate
python .github/scripts/per_category_coco_eval.py \
  --gt "$DATA/test/annotation_merged.json" \
  --pred "$REPO/experiments/<new_experiment_dir>/dumps/pins/coco_predictions_segm.json" \
  --out "$REPO/experiments/<new_experiment_dir>/per_category_eval_summary.json"
'
```

Compare the new `thick_wire`/`thin_wire`/`gate_pad` AP against the pre-fix
baseline (0.369 / 0.242 / 0.188) to confirm the fix actually improves them.

## Step 11 -- FIX CONFIRMED WORKING

Job `238099599` completed successfully (`Successfully completed.`, ~12h30m,
fresh from base `sam3.pt` on the corrected `annotation_merged.json` files,
zero `WEIGHT-NaN` hits throughout). Re-ran the per-category breakdown against
the new predictions:

| Category | AP before fix | AP after fix | Delta |
|---|---|---|---|
| et_pin | 0.913 | 0.857 | -0.056 |
| eberhard_pin | 0.969 | 0.951 | -0.018 |
| wuxi_pin | 0.955 | 0.932 | -0.024 |
| matrix_chip | 0.961 | 0.951 | -0.010 |
| **thick_wire** | **0.369** | **0.896** | **+0.527** |
| **thin_wire** | **0.242** | **0.675** | **+0.433** |
| **gate_pad** | **0.188** | **0.573** | **+0.385** |
| war_crack | -- | -- | still unscoreable (0 test instances) |
| **Overall (aggregate)** | **0.657** | **0.833** | **+0.176** |

**The 3 previously-broken categories improved 2.4x-3x**, decisively confirming
the split-photo contradictory-negatives bug (Step 9) was the primary driver of
their poor AP -- not object geometry as originally hypothesized (Step 4). The
5 original categories moved only slightly (a few points down), consistent
with normal run-to-run variance from training a fresh model from scratch
rather than resuming from a checkpoint.

`war_crack`'s train/test split gap (0 test instances) is unrelated to this
fix and remains open -- tracked as a separate todo item.

## Step 12 -- Computer vision expert panel: mask quality still poor for thick_wire/thin_wire

Even after Step 11's fix, visual review of the per-category overlay posters
(`sam3/eval/per_category_eval.py`) showed the underlying SEGMENTATION masks
for `thick_wire`/`thin_wire` were still poor in a specific way:

- **Blobby, imprecise boundaries** -- predicted masks spilling into
  background/chip-pad regions rather than tightly tracing the wire.
- **Mask fragmentation** -- a single wire instance's predicted mask sometimes
  broken into two disconnected pieces instead of one continuous region.

Consulted a "computer vision / deep learning segmentation expert panel"
framing to reason through root causes and fixes:

1. **ViT-backbone resolution expert**: at `scratch.resolution=1008` with ViT
   `patch_size=16` -> a 63x63 token grid. `thick_wire` avg width ~37px (~2.3
   patches), `thin_wire` avg width ~26px (~1.6 patches) -- the wire's entire
   cross-section fits inside 1-2 backbone tokens, an architectural resolution
   ceiling independent of how much more data is added.
2. **Loss-function expert**: standard Dice/BCE losses barely penalize
   boundary imprecision OR connectivity breaks on thin objects, since their
   total pixel count is tiny relative to the image. The literature's answer
   for exactly this failure mode (thin, tubular/curvilinear structures --
   vessels, roads, wires) is **clDice** (Shit et al., CVPR 2021): a
   topology-preserving loss based on skeleton (centerline) recall/precision
   rather than raw pixel overlap.
3. **Training-strategy expert**: "more images" helps generalization but not
   necessarily boundary crispness given the resolution ceiling above --
   `thick_wire` already has MORE training instances (4810) than several 90%+
   AP categories, so volume alone wasn't the limiting factor.

**Decision**: implement clDice as the most targeted, well-established fix,
with an easy config-level revert path back to the original default losses.

### The clDice loss implementation (first version)

`soft_cldice_loss()` (added to `sam3/train/loss/loss_fns.py`) scores based on
the predicted/GT mask's *skeleton* instead of raw pixel overlap. If a
predicted mask is fragmented into two disconnected pieces, the gap's portion
of the GT skeleton is not "recalled" by the prediction -- directly and
heavily penalizing the loss. Skeletonization is done via iterative
differentiable morphological erosion/dilation (max-pooling based), the
standard technique from the clDice paper's reference implementation:

```python
def _soft_erode(img: torch.Tensor) -> torch.Tensor:
    \"\"\"Differentiable morphological erosion via min-pooling (implemented as
    negated max-pooling), applied separably along H then W (3x1 then 1x3).
    `img`: (N, C, H, W).
    \"\"\"
    p1 = -F.max_pool2d(-img, kernel_size=(3, 1), stride=(1, 1), padding=(1, 0))
    p2 = -F.max_pool2d(-img, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
    return torch.min(p1, p2)


def _soft_dilate(img: torch.Tensor) -> torch.Tensor:
    \"\"\"Differentiable morphological dilation via max-pooling. `img`: (N, C, H, W).\"\"\"
    return F.max_pool2d(img, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))


def _soft_open(img: torch.Tensor) -> torch.Tensor:
    \"\"\"Morphological opening = erosion followed by dilation.\"\"\"
    return _soft_dilate(_soft_erode(img))


def soft_skeletonize(img: torch.Tensor, iterations: int = 10) -> torch.Tensor:
    \"\"\"Iteratively approximate the morphological skeleton (centerline) of a
    soft/probabilistic mask `img` (N, C, H, W), fully differentiable.
    \"\"\"
    img1 = _soft_open(img)
    skel = F.relu(img - img1)
    for _ in range(iterations):
        img = _soft_erode(img)
        img1 = _soft_open(img)
        delta = F.relu(img - img1)
        skel = skel + F.relu(delta - skel * delta)
    return skel


def _soft_cldice_loss(inputs, targets, num_boxes, iterations, smooth, reduce):
    \"\"\"inputs: raw mask logits (N,1,H,W). targets: binary {0,1} masks (N,1,H,W).\"\"\"
    inputs = inputs.sigmoid()
    skel_pred = soft_skeletonize(inputs, iterations)
    skel_true = soft_skeletonize(targets, iterations)

    # precision: fraction of the predicted skeleton that lies inside the GT mask
    tprec = (skel_pred * targets).sum(dim=(1, 2, 3)) / (skel_pred.sum(dim=(1, 2, 3)) + smooth)
    # sensitivity/recall: fraction of the GT skeleton that lies inside the predicted mask
    # -- this is the term that penalizes fragmentation/disconnection directly.
    tsens = (skel_true * inputs).sum(dim=(1, 2, 3)) / (skel_true.sum(dim=(1, 2, 3)) + smooth)

    cl_dice = 2.0 * (tprec * tsens) / (tprec + tsens + smooth)
    loss = 1 - cl_dice
    loss = torch.nan_to_num(loss, nan=0.0, posinf=10.0, neginf=0.0)
    if not reduce:
        return loss
    return loss.sum() / max(num_boxes, 1)
```

Wired into the `Masks` loss class via a new `weight_dict["loss_cldice"]` entry
(new `Masks.compute_cldice` property, only active in the non-point-sampled
path since it needs the full 2D mask to skeletonize). `full_train_config.yaml`
sets `loss_cldice: 15.0` by default. **To revert to the original default
losses (mask + dice only): just remove/comment out the `loss_cldice` line --
no code change needed.**

1. ✅ **DONE** -- Root cause confirmed (Step 9): split-photo contradictory
   negatives affecting 100% of the new-category data.
2. ✅ **DONE** -- Fix implemented and verified (Step 10): duplicate-photo
   entries merged, zero annotation loss, config updated to use the fixed
   files.
3. ✅ **DONE** -- Fix confirmed working (Step 11): fresh training run on the
   corrected annotations shows the 3 new categories improved 2.4x-3x AP.
4. ⏳ **NOT STARTED** -- Separately, `war_crack`'s train/test split gap (0 test
   instances) still needs a fix before it can be evaluated at all -- unrelated
   to this investigation, tracked as its own todo item.
5. Also added `sam3/eval/per_category_eval.py` (`PerCategoryEvalWithOverlays`)
   so future training runs get this per-category breakdown + overlay JPEGs
   automatically every validation epoch, without needing to manually re-run
   the standalone script.

### Validated directionally with a synthetic fragmented-vs-connected test

Before wiring into training, tested `soft_cldice_loss` against `dice_loss` on
synthetic masks (perfect match vs. the same mask with a gap cut into it,
fragmenting it into two pieces):

```python
# realistic thin_wire width (~26px) and thick_wire width (~37px) at full 1008 resolution
for width, name in [(26, "thin_wire-like"), (37, "thick_wire-like")]:
    gt = torch.zeros(2, 1, H, W)
    gt[:, 0, 400:400+width, 100:900] = 1.0

    pred_perfect = (gt.clone() * 2 - 1) * 8
    pred_frag = gt.clone()
    pred_frag[:, 0, 400:400+width, 480:520] = 0.0  # 40px gap in the middle
    pred_frag_logits = (pred_frag * 2 - 1) * 8

    d_perfect = dice_loss(pred_perfect.flatten(1), gt.flatten(1), num_boxes=2)
    d_frag = dice_loss(pred_frag_logits.flatten(1), gt.flatten(1), num_boxes=2)
    cl_perfect = soft_cldice_loss(pred_perfect, gt, num_boxes=2)
    cl_frag = soft_cldice_loss(pred_frag_logits, gt, num_boxes=2)
```

Result:

```
--- thin_wire-like (width=26px) ---
  Dice   -- perfect: 0.0081  fragmented: 0.0338  delta: 0.0256
  clDice -- perfect: 0.3401  fragmented: 0.3628  delta: 0.0227
--- thick_wire-like (width=37px) ---
  Dice   -- perfect: 0.0057  fragmented: 0.0314  delta: 0.0256
  clDice -- perfect: 0.3409  fragmented: 0.3639  delta: 0.0231
```

Both losses correctly increase when the mask is fragmented (same directional
behavior, comparable delta magnitude). clDice's higher baseline (~0.34 even
for a "perfect" match) reflects that the soft-skeleton is an *approximation*
of the true morphological skeleton (converges to a slightly-thicker-than-1px
centerline for these mask thicknesses) -- this doesn't hurt training since
what matters is the correct direction/gradient signal, not a calibrated
absolute value.

## Step 13 -- First training attempt OOM'd (host memory, not GPU)

Submitted the first validation run (job `238284710`) with `loss_cldice: 15.0`
active. It was **killed after ~80 seconds**:

```
TERM_MEMLIMIT: job killed after reaching LSF memory usage limit.
Max Memory : 65536 MB   (Total Requested Memory: 200.00 MB)
```

Critically, this is **host RAM**, not GPU memory (`Mem (GB): 46.00/46.00` in
the training log looked normal for the GPU side). Root cause: soft-
skeletonization keeps ~10 iterations worth of **full-resolution** (up to
1008x1008) intermediate activations alive for the backward pass, and the
original implementation had a `try/except torch.OutOfMemoryError` fallback
that moved the same huge tensor to host RAM on a GPU OOM -- for a batch with
potentially hundreds of mask instances (`category_chunk_size=8`), this
fallback itself blew past the tiny ~200MB host-memory reservation this
GPU-bound job type normally needs.

### The downsampling fix

Skeleton topology (connectivity, centerline shape) doesn't need full pixel
resolution to be a useful loss signal. Fixed by downsampling both masks to a
small fixed size (128x128) **before** skeletonizing, and removing the risky
CPU fallback entirely (better to fail fast on a genuine GPU OOM than silently
balloon host RAM on an undersized reservation):

```python
def soft_cldice_loss(
    inputs,
    targets,
    num_boxes,
    iterations: int = 10,
    smooth: float = 1.0,
    reduce=True,
    downsample_size: int = 128,
):
    """
    downsample_size: soft-skeletonization keeps ~10 iterations worth of full-size
        intermediate activations alive for backward (autograd), which at the full
        mask resolution (up to ~1008x1008, with potentially hundreds of instances
        per batch given category_chunk_size) can blow past the per-job HOST memory
        reservation (this HPC's GPU-bound jobs are only allotted a small host-RAM
        reservation, since GPU-only compute normally needs very little). Both
        `inputs` and `targets` are downsampled to `downsample_size x downsample_size`
        BEFORE skeletonizing -- cuts memory/compute by roughly
        (H*W) / downsample_size**2, e.g. ~62x at H=W=1008, downsample_size=128.
        Set to `None` to disable (use full resolution) if you have memory headroom
        to spare.
    """
    if downsample_size is not None and inputs.shape[-1] > downsample_size:
        inputs = F.interpolate(
            inputs, size=(downsample_size, downsample_size), mode="bilinear", align_corners=False
        )
        targets = F.interpolate(
            targets, size=(downsample_size, downsample_size), mode="nearest"
        )
    return _soft_cldice_loss(inputs, targets, num_boxes, iterations, smooth, reduce)
```

**Re-validated the fix directly** at realistic scale before resubmitting --
ran `soft_cldice_loss` on a batch of 300 instances at FULL 1008x1008
resolution (matching a worst-case real batch) to confirm no memory blowup,
then re-ran the perfect-vs-fragmented sensitivity test above (still passes:
clDice correctly and proportionally penalizes fragmentation after
downsampling to 128x128).

### Re-submitted and confirmed stable

Killed the crashed job, resubmitted (same `bsub` params, no `rusage[mem=]`
change needed -- a normal successful run without clDice previously used
~33GB host RAM comfortably, well under whatever limit the OOM actually
tripped at ~64GB) as job `238286332`. Confirmed running well past the
previous crash point (50+ steps, GPU memory steady at 44GB matching the
known-tight-config baseline, no crash) before leaving it to run overnight.

**Generalizable lesson**: any new loss/metric that keeps extra tensors alive
for backward (especially anything involving multiple iterations of an
operation, like iterative skeletonization) needs an explicit memory-scaling
plan BEFORE being wired into a batch of potentially hundreds of full-resolution
instances -- test at realistic worst-case batch size/resolution, not just a
toy example, before committing to a multi-hour training run.

## Open items / next steps

1. ✅ **DONE** -- Root cause confirmed (Step 9): split-photo contradictory
   negatives affecting 100% of the new-category data.
2. ✅ **DONE** -- Fix implemented and verified (Step 10): duplicate-photo
   entries merged, zero annotation loss, config updated to use the fixed
   files.
3. ✅ **DONE** -- Fix confirmed working (Step 11): fresh training run on the
   corrected annotations shows the 3 new categories improved 2.4x-3x AP.
4. ✅ **DONE** -- clDice loss added (Steps 12-13) to address remaining
   blobby-boundary and mask-fragmentation issues on `thick_wire`/`thin_wire`,
   with a memory-safe downsampling fix after an initial OOM.
5. 🔄 **IN PROGRESS** -- Overnight validation run (job `238286332`, fresh
   `experiments/8cat_bondwire_cldice` dir, starts clean from base `sam3.pt`)
   to measure whether clDice actually improves `thick_wire`/`thin_wire` mask
   quality (both per-category AP and, more importantly for this specific
   fix, visual boundary/connectivity quality in the overlay posters).
6. ⏳ **NOT STARTED** -- Separately, `war_crack`'s train/test split gap (0 test
   instances) still needs a fix before it can be evaluated at all -- unrelated
   to this investigation, tracked as its own todo item.
7. Update this notebook with job `238286332`'s per-category AP + overlay
   poster results once the run completes.
